In [1]:
# from featup.util import norm
import torch
import torchvision.transforms as T
import torch.nn.functional as F
from tools.utils import load_yaml_config, instantiate_from_config, seed_everything
from tools.utils import plot_traj
import matplotlib.pyplot as plt

import einops
from PIL import Image
from tqdm import tqdm

# Paths for things to load in

In [2]:
image_path = 'demo_img2.jpg' # change this to your image path
checkpoint_pth = 'demo_weights.pth'
config_pth = 'demo_config.yaml'

# Load in config file and extract relevant hyperparmeters
Note that the config file is general and has all the various paramaters we need for inference AND training. Feel free to check the demo config file and change parameters there, or change them in the cell below.

In [3]:
config = load_yaml_config(config_pth)
num_samples = config['val_dataset']['params']['num_samples'] # how many trajectories to generate
seq_len = config['trainer']['seq_len_to_gen_for_eval'] # length of each trajectory
cfg_on = config['trainer']['params']['cfg_on'] # whether to use classifier free guidance
cfg_scale = config['trainer']['params']['cfg_scale'] # classifier free guidance scale
eta = config['diff_val_scheduler']['eta'] # DDIM eta parameter
model_name = config['model']['target'] # model class name, don't change this since it corresponds to the demo weights

# Set the seed for reproducibility
Change the seed to get different traejctories, or remove it to get random ones every run.

In [4]:
seed = 12
seed_everything(seed)

Random seed set as 12


# Set the device

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Setup the model

In [6]:
model = instantiate_from_config(config['model']).to(device)
checkpoint = torch.load(checkpoint_pth, map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])

/tmp/ipykernel_2092428/310192490.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_pth, map_location='cpu')


<All keys matched successfully>

# Setup scheduler

In [7]:
scheduler = instantiate_from_config(config['diff_val_scheduler'])
if config['diff_val_scheduler']['target'] == 'diffusers.schedulers.DDIMScheduler':
    scheduler.set_timesteps(config['diff_val_scheduler']['num_inference_steps'])

/home/nisar2/anaconda3/envs/diffeye2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Get image and extract featup features

In [8]:
# Getting the image:
pil_img = Image.open(image_path).convert("RGB")
original_h, original_w = pil_img.size[1], pil_img.size[0]
transform = T.Compose([
    T.Resize((224,224), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    norm
]) # transformations for image
image_tensor = transform(pil_img).unsqueeze(0).to(device)

# Extract FeatUp features:
use_norm = False
upsampler = torch.hub.load("mhamilton723/FeatUp", 'saliencyv2', use_norm=use_norm).to(device)
hr_feats = upsampler(image_tensor)
hr_feats_interpolated = F.interpolate(hr_feats, size=(32,32), mode='bilinear', align_corners=False)
hr_feats_interpolated = einops.rearrange(hr_feats_interpolated, "b n h w -> b (h w) n") 

Using cache found in /home/nisar2/.cache/torch/hub/mhamilton723_FeatUp_main
/home/nisar2/Desktop/diffeye/featup/featurizers/saliencyv2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/nisar2/Desktop/diffeye/featup/featurizers/saliencyv2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/nisar2/Desktop/diffeye/featup/featurizers/saliencyv2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Using cache found in /home/nisar2/.cache/torch/hub/facebookresearch_saliencyv2_main
/home/nisar2/.cache/torch/hub/facebookresearch_saliencyv2_main/saliencyv2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/nisar2/.cache/torch/hub/facebookresearch_saliencyv2_main/saliencyv2/layers/at

# Generating the trajectory

In [9]:
# start with pure noise
generated_trajs = torch.randn((1,2,seq_len)).to(device) # (current_batch_size, 2, seq_len)

model.eval()
with torch.no_grad():
    # Run the correct model
    if model_name == 'models.unet_saliency.Unet' or model_name == 'models.unet_saliency_hires.Unet':
        print(f'Generating trajectories using {model_name}')
        if cfg_on: # using classifier free guidance
            print('Using classifier free guidance')
            # iterate through the diffusion timesteps
            for i in tqdm(scheduler.timesteps, desc='Generating trajectories with classifier free guidance', total=len(scheduler.timesteps)):
                noise_pred_w_cond, _ = model(generated_trajs, torch.tensor([i]).to(device), hr_feats_interpolated)
                noise_pred_wo_cond, _ = model(generated_trajs, torch.tensor([i]).to(device), torch.zeros_like(hr_feats_interpolated).to(device))
                noise_pred = (1.0 - cfg_scale) * noise_pred_wo_cond + cfg_scale * noise_pred_w_cond

                scheduler_output = scheduler.step(
                    model_output=noise_pred,
                    timestep=i,
                    sample=generated_trajs,
                    eta=eta,
                )
                generated_trajs = scheduler_output.prev_sample
        else: # not using classifier free guidance
            print('Not using classifier free guidance')
            # iterate through the diffusion timesteps
            for i in tqdm(scheduler.timesteps, desc='Generating trajectories without classifier free guidance', total=len(scheduler.timesteps)):
                noise_pred, _ = model(generated_trajs, torch.tensor([i]).to(device), hr_feats_interpolated)

                scheduler_output = scheduler.step(
                    model_output=noise_pred,
                    timestep=i,
                    sample=generated_trajs,
                    eta=eta,
                )
                generated_trajs = scheduler_output.prev_sample
    else: # unsupported model
        raise ValueError(f"Model {model_name} is not supported")

    # detach the generated trajectories from the graph and move them to cpu
    generated_trajs = generated_trajs.detach().cpu() # (current_batch_size, 2, seq_len)

    # get the width and height of the stimulus
    # this is the size of the stimulus after it has been resized and will be the same for all stimuli (ie 224 by 224)
    w, h = 224, 224

    # compute centers
    center_x = w // 2 
    center_y = h // 2

    # unnormalize the ground truth gaze points
    unnormed_generated_trajs = generated_trajs * torch.tensor([center_x, center_y])[torch.newaxis, :, torch.newaxis] + \
                torch.tensor([center_x, center_y])[torch.newaxis, :, torch.newaxis] # (current_batch_size, 2, seq_len)

    # resize the generated trajectories to original image size
    og_generated_traj = unnormed_generated_trajs[i].clone()
    og_generated_traj[0,:] *= original_w / 224
    og_generated_traj[1,:] *= original_h / 224

    # plot the trajectory on the image
    fig, ax = plt.subplots()
    plot_traj(og_generated_traj.T, pil_img, stim_first=False, marker_size=1, ax=ax)
    plt.show()

Generating trajectories using models.unet_saliency_hires.Unet
Using classifier free guidance


Generating trajectories with classifier free guidance: 100%|██████████| 50/50 [00:01<00:00, 27.48it/s]
